# ML-09 — Validation and Research Claim Audit

This notebook audits two findings from the FlyRank SEO research paper and then stress-tests my Week-5 next-day click model. Language is observational and decision-support only.


## 1. Two paper findings + my methodology questions

**Finding 1 — growing content tends to be younger and longer.** Growing pages averaged about 3.2K words and 184 days old, versus 2.3K words and 230 days for declining pages. The paper calls this observational. 

**Methodology question:** where exactly does the growth/decline label come from, and does the analysis control for page or brand differences that could make pages both younger/longer and growing? The label is based on observed performance, so association does not establish that writing more or reducing age causes growth.

**Finding 2 — performance peaks around 61–90 days and declines later.** The paper reports a health score around 33.1 at 61–90 days versus 14 at 271–365 days, while warning that refresh activity and survivor bias matter. 

**Methodology question:** does validation separate age from refresh activity, publication timing, and survivor bias? Older pages may differ because the pages that survive and get refreshed are systematically different. These are constructive checks, not a rejection of the findings.


## 2. My model under an honest split (before/after)

Before = a random row split, which can place the same client in train and test. After = GroupShuffleSplit by client, which tests unseen-client generalization. Both use March 1–20 daily rows, the same four features, the same next-day label, and ROC-AUC.


In [1]:
from pathlib import Path
import os,duckdb,pandas as pd,numpy as np
from sklearn.model_selection import train_test_split,GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score,average_precision_score

p=list((Path.home()/'.cache'/'huggingface'/'hub').rglob('fact_content_daily_performance/month=2026-03/data_0.parquet'))
if p: 
   path=str(p[0])
else:
    try:
        from google.colab import userdata; 
        token=userdata.get('HF_TOKEN')
    except Exception: 
       token=os.environ.get('HF_TOKEN') or os.environ.get('HUGGINGFACE_HUB_TOKEN')
    if not token: 
       raise RuntimeError('Set HF_TOKEN as a Colab Secret or environment variable; never paste it here.')
    
    from huggingface_hub import hf_hub_download; 
    path=hf_hub_download('FlyRank/internship-warehouse','fact_content_daily_performance/month=2026-03/data_0.parquet',repo_type='dataset',token=token)

safe=path.replace(chr(39),chr(39)*2); 
con=duckdb.connect(); 

con.execute(f"CREATE VIEW march AS SELECT * FROM read_parquet('{safe}')")
sql="""WITH p AS (SELECT report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,NULLIF(gsc_avg_position,0) pos,LEAD(gsc_clicks) OVER(PARTITION BY client_hash_id,content_hash_id ORDER BY report_date) nxt FROM march WHERE gsc_data_available IS TRUE) SELECT report_date,client_hash_id,content_hash_id,log(1+gsc_impressions) log_impressions,log(1+gsc_clicks) log_clicks,COALESCE(pos,100.0) avg_position,COALESCE(gsc_clicks::DOUBLE/NULLIF(gsc_impressions,0),0.0) ctr,(nxt>0)::INTEGER y FROM p WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-20' AND nxt IS NOT NULL USING SAMPLE 250000 ROWS (reservoir,202609)"""
df=con.sql(sql).df(); 

features=['log_impressions','log_clicks','avg_position','ctr']; 
print('Rows:',len(df),'base rate:',round(df.y.mean(),4)); 

X=df[features]; y=df.y
ri,rj=train_test_split(np.arange(len(df)),test_size=.25,random_state=202609,stratify=y); 
gi,gj=next(GroupShuffleSplit(n_splits=1,test_size=.25,random_state=202609).split(X,y,groups=df.client_hash_id))

def fit_score(a,b):
 m=make_pipeline(StandardScaler(),LogisticRegression(max_iter=300,random_state=202609)); 
 m.fit(X.iloc[a],y.iloc[a]); 
 s=m.predict_proba(X.iloc[b])[:,1]; 
 return m,s

rm,rs=fit_score(ri,rj); 
gm,gs=fit_score(gi,gj); 
comparison=pd.DataFrame([{'split':'Random rows (before)','clients_overlap':len(set(df.client_hash_id.iloc[ri])&set(df.client_hash_id.iloc[rj])),'ROC_AUC':roc_auc_score(y.iloc[rj],rs),'average_precision':average_precision_score(y.iloc[rj],rs)},{'split':'Grouped by client (honest after)','clients_overlap':len(set(df.client_hash_id.iloc[gi])&set(df.client_hash_id.iloc[gj])),'ROC_AUC':roc_auc_score(y.iloc[gj],gs),'average_precision':average_precision_score(y.iloc[gj],gs)}]); 
display(comparison.round(4))


Rows: 154176 base rate: 0.1191


,split,clients_overlap,ROC_AUC,average_precision
0,Random rows (before),40,0.8737,0.5532
1,Grouped by client (honest after),0,0.8725,0.5150


## 3. Leakage audit

The final feature set is `log_impressions`, `log_clicks`, `avg_position`, and `ctr`. These are calculated from day *t* only. The label is `y = next_day_has_click`, computed from day *t+1*. I excluded client/content IDs, product flags, all next-day fields, trend/label columns, and June’s final-month sample.


In [2]:
forbidden=['next','future','label','trend','flag','health','priority']; 
print('Feature audit:'); 
display(pd.DataFrame({'feature':features,'contains_forbidden_token':[any(w in f.lower() for w in forbidden) for f in features],'available_before_label':[True]*len(features)})); 
assert not any(any(w in f.lower() for w in forbidden) for f in features)

# Deliberate audit attack: a copy of y must produce near-perfect ranking, proving the harness can detect leakage.
leaky=roc_auc_score(y.iloc[gj],y.iloc[gj]); 
honest=roc_auc_score(y.iloc[gj],gs); 

print(f'Intentional label-copy AUC (invalid): {leaky:.4f}'); 
print(f'Honest grouped AUC: {honest:.4f}'); 
print('The perfect label-copy score is rejected and is not used as a feature.')


Feature audit:


,feature,contains_forbidden_token,available_before_label
0,log_impressions,False,True
1,log_clicks,False,True
2,avg_position,False,True
3,ctr,False,True


Intentional label-copy AUC (invalid): 1.0000
Honest grouped AUC: 0.8725
The perfect label-copy score is rejected and is not used as a feature.


## 4. Claim rewrite

**Too-strong original claim:** “The model identifies pages that will get more clicks.”

**Evidence-aligned rewrite:** “On the March 2026 slice, the model measured directional ranking skill for the observed next-day click proxy on unseen clients. This is decision support for prioritization; it does not prove that a recommendation causes more clicks or generalizes to every month.”

Failure examples are expected: pages with sparse impressions and volatile day-to-day demand can be false positives or false negatives even when the inputs are clean.


In [3]:
pred=gs>=np.quantile(gs,.9); 
errors=np.select([pred&(y.iloc[gj].to_numpy()==0),(~pred)&(y.iloc[gj].to_numpy()==1)],['false_positive','false_negative'],'correct'); 
err=df.iloc[gj].copy(); 
err['score']=gs; 
err['error_type']=errors; 

print('Error counts:'); 
display(pd.Series(errors).value_counts().to_frame('n')); 
print('Three real failure examples (pseudonymous IDs only):'); 
display(err[err.error_type!='correct'][['client_hash_id','content_hash_id','log_impressions','log_clicks','avg_position','ctr','y','score','error_type']].head(3))


Error counts:


,n
correct,17100
false_positive,979
false_negative,960


Three real failure examples (pseudonymous IDs only):


,client_hash_id,content_hash_id,log_impressions,log_clicks,avg_position,ctr,y,score,error_type
24,client_e547b89c05043229,content_8ca42a6795568982,2.037426,0.30103,0.444444,0.009259,0,0.393801,false_positive
166,client_e547b89c05043229,content_243c9559f0238522,2.217484,0.00000,0.963415,0.000000,0,0.276375,false_positive
202,client_157ffe4d4a595515,content_08dca348aa3fcda6,1.755875,0.00000,4.178571,0.000000,1,0.131827,false_negative


## 5. Self-check

- [x] Two paper findings with constructive label and validation questions
- [x] Random before/grouped-after comparison on the same March data and metric
- [x] Feature leakage audit plus deliberate label-copy attack
- [x] Concrete errors and a cautious claim rewrite
